In [1]:
import pandas as pd

df = pd.read_csv("end-to-end/report_complete_processed.csv")

In [2]:
df.columns

Index(['trace_name', 'number_of_requests', 'wsr', 'slab_size', 'slab_cnt',
       'rebalanceDiffRatio', 'mhMinDiff', 'rebalance_strategy', 'allocator',
       'tag', 'throughput', 'rebalanced_slabs', 'miss_ratio',
       'n_rebalanced_slabs', 'monitor_interval', 'n_alloc_failures', 'uuid',
       'mhMovingAverageParam', 'mhMinDiff.1', 'thresholdAIADStep', 'emrLow',
       'emrHigh', 'base_dir', 'miss_ratio_reduction_from_disabled',
       'miss_ratio_reduction_from_lru_disabled', 'tuned_improvement'],
      dtype='object')

In [3]:
twitter_result = df[df['trace_name'].str.startswith("twitter")].copy()

In [ ]:
"""
twitter_result, 
filter allocator = LRU
group by 
trace_name, wsr 
within each group find the row rebalance_strategy = marginal-hits-tuned
also the row rebalance_strategy = lama

get the miss ratio of the two, name them marginal_hits_tuned_miss_ratio and lama_miss_ratio
also compute the difference between the two, name it miss_ratio_diff
"""

In [11]:
twitter_result[(twitter_result['rebalance_strategy'] == 'lama') &
               (twitter_result['wsr'] == 0.01)]['miss_ratio_reduction_from_disabled'].describe()

count    2.300000e+01
mean     4.834161e-02
std      7.779886e-02
min      0.000000e+00
25%      1.597205e-09
50%      4.538539e-03
75%      7.518772e-02
max      3.279959e-01
Name: miss_ratio_reduction_from_disabled, dtype: float64

In [14]:
twitter_result[(twitter_result['rebalance_strategy'] == 'marginal-hits-tuned') & (twitter_result['allocator'] == 'LRU') &
               (twitter_result['wsr'] == 0.01)]['miss_ratio_reduction_from_disabled'].describe()

count    3.600000e+01
mean     2.769351e-02
std      5.847295e-02
min     -1.113298e-02
25%      5.586909e-10
50%      1.893582e-03
75%      2.851740e-02
max      2.975456e-01
Name: miss_ratio_reduction_from_disabled, dtype: float64

In [19]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_twitter_boxplot_plotly(twitter_result, wsr, width=1000, height=600):
    """
    Create an interactive boxplot using Plotly showing miss_ratio_reduction_from_lru_disabled 
    by allocator, colored by rebalance_strategy for a specific WSR value.
    
    Parameters:
    twitter_result: DataFrame with Twitter trace results
    wsr: float, the WSR value to filter on
    width: int, figure width in pixels
    height: int, figure height in pixels
    """
    # Filter data for the specified WSR
    wsr_data = twitter_result[twitter_result['wsr'] == wsr].copy()
    
    if wsr_data.empty:
        print(f"No data found for WSR = {wsr}")
        return
    
    # Create the boxplot using Plotly Express
    fig = px.box(wsr_data, 
                 x='allocator', 
                 y='miss_ratio_reduction_from_lru_disabled',
                 color='rebalance_strategy',
                 title=f'Miss Ratio Reduction from LRU Disabled by Allocator (WSR = {wsr})',
                 labels={
                     'allocator': 'Allocator',
                     'miss_ratio_reduction_from_lru_disabled': 'Miss Ratio Reduction from LRU Disabled',
                     'rebalance_strategy': 'Rebalancing Strategy'
                 },
                 color_discrete_sequence=px.colors.qualitative.Set2,
                 points=False)  # We'll add mean markers separately
    
    # Add mean markers for each box
    allocators = sorted(wsr_data['allocator'].unique())
    strategies = sorted(wsr_data['rebalance_strategy'].unique())
    
    # Get the color mapping from the boxplot
    colors = px.colors.qualitative.Set2
    strategy_colors = {strategy: colors[i % len(colors)] for i, strategy in enumerate(strategies)}
    
    # Calculate box positions (similar to how plotly positions them)
    box_width = 0.8 / len(strategies)  # Total width divided by number of strategies
    
    for i, allocator in enumerate(allocators):
        for j, strategy in enumerate(strategies):
            subset = wsr_data[(wsr_data['allocator'] == allocator) & 
                             (wsr_data['rebalance_strategy'] == strategy)]
            
            if not subset.empty:
                mean_val = subset['miss_ratio_reduction_from_lru_disabled'].mean()
                
                # Calculate x position for this box
                base_x = i
                offset = (j - (len(strategies) - 1) / 2) * box_width
                x_pos = base_x + offset
                
                # Add mean marker
                fig.add_trace(go.Scatter(
                    x=[x_pos],
                    y=[mean_val],
                    mode='markers',
                    marker=dict(
                        symbol='diamond',
                        size=8,
                        color='white',
                        line=dict(color=strategy_colors[strategy], width=2)
                    ),
                    name=f'Mean ({strategy})',
                    showlegend=False,  # Don't clutter the legend
                    hovertemplate=f'<b>Mean</b><br>Allocator: {allocator}<br>Strategy: {strategy}<br>Mean: %{{y:.4f}}<extra></extra>'
                ))
    
    # Update layout for better appearance
    fig.update_layout(
        title={
            'text': f'Miss Ratio Reduction from LRU Disabled by Allocator (WSR = {wsr})',
            'x': 0.5,
            'font': {'size': 18, 'family': 'Arial, sans-serif'}
        },
        xaxis_title={
            'text': 'Allocator',
            'font': {'size': 14, 'family': 'Arial, sans-serif'}
        },
        yaxis_title={
            'text': 'Miss Ratio Reduction from LRU Disabled',
            'font': {'size': 14, 'family': 'Arial, sans-serif'}
        },
        legend_title={
            'text': 'Rebalancing Strategy',
            'font': {'size': 12, 'family': 'Arial, sans-serif'}
        },
        width=width,
        height=height,
        showlegend=True,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    # Update axes
    fig.update_xaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=1,
        linecolor='black'
    )
    
    fig.update_yaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=1,
        linecolor='black'
    )
    
    # Show the plot
    fig.show()
    
    # Print some summary statistics
    print(f"\nSummary for WSR = {wsr}:")
    print(f"Number of data points: {len(wsr_data)}")
    print(f"Allocators: {sorted(wsr_data['allocator'].unique())}")
    print(f"Rebalancing strategies: {sorted(wsr_data['rebalance_strategy'].unique())}")
    
    # Print mean values for each combination
    print("\nMean values by Allocator and Strategy:")
    for allocator in sorted(wsr_data['allocator'].unique()):
        print(f"\n{allocator}:")
        for strategy in sorted(wsr_data['rebalance_strategy'].unique()):
            subset = wsr_data[(wsr_data['allocator'] == allocator) & 
                             (wsr_data['rebalance_strategy'] == strategy)]
            if not subset.empty:
                mean_val = subset['miss_ratio_reduction_from_lru_disabled'].mean()
                print(f"  {strategy}: {mean_val:.6f}")
    
    return fig

# Example usage:
# plot_twitter_boxplot_plotly(twitter_result, 0.01)
# plot_twitter_boxplot_plotly(twitter_result, 0.1)

In [20]:
plot_twitter_boxplot_plotly(twitter_result, 0.01)


Summary for WSR = 0.01:
Number of data points: 649
Allocators: ['LRU', 'LRU2Q', 'TINYLFU']
Rebalancing strategies: ['disabled', 'eviction-rate', 'hits', 'lama', 'marginal-hits', 'marginal-hits-tuned', 'tail-age']

Mean values by Allocator and Strategy:

LRU:
  disabled: 0.000000
  eviction-rate: 0.014525
  hits: 0.025970
  lama: 0.048342
  marginal-hits: 0.028228
  marginal-hits-tuned: 0.027694
  tail-age: 0.010874

LRU2Q:
  disabled: 0.010815
  eviction-rate: 0.024902
  hits: 0.034953
  marginal-hits: 0.033691
  marginal-hits-tuned: 0.042235
  tail-age: 0.020193

TINYLFU:
  disabled: 0.002587
  eviction-rate: 0.014342
  hits: 0.029343
  marginal-hits: 0.025312
  marginal-hits-tuned: 0.030395
  tail-age: 0.016692


In [4]:
# Filter for LRU allocator only
lru_twitter = twitter_result[twitter_result['allocator'] == 'LRU'].copy()

# Group by trace_name and wsr, then process each group
comparison_results = []

for (trace_name, wsr), group in lru_twitter.groupby(['trace_name', 'wsr']):
    # Find marginal-hits-tuned row
    marginal_hits_tuned_row = group[group['rebalance_strategy'] == 'marginal-hits-tuned']
    
    # Find lama row
    lama_row = group[group['rebalance_strategy'] == 'lama']
    
    # Only process if both strategies exist for this trace_name, wsr combination
    if len(marginal_hits_tuned_row) > 0 and len(lama_row) > 0:
        marginal_hits_tuned_miss_ratio = marginal_hits_tuned_row['miss_ratio'].iloc[0]
        lama_miss_ratio = lama_row['miss_ratio'].iloc[0]
        miss_ratio_diff = marginal_hits_tuned_miss_ratio - lama_miss_ratio
        
        comparison_results.append({
            'trace_name': trace_name,
            'wsr': wsr,
            'marginal_hits_tuned_miss_ratio': marginal_hits_tuned_miss_ratio,
            'lama_miss_ratio': lama_miss_ratio,
            'miss_ratio_diff': miss_ratio_diff
        })

# Convert to DataFrame
comparison_df = pd.DataFrame(comparison_results)
print(f"Found {len(comparison_df)} trace_name-wsr combinations with both strategies")
comparison_df.head()

Found 59 trace_name-wsr combinations with both strategies


,trace_name,wsr,marginal_hits_tuned_miss_ratio,lama_miss_ratio,miss_ratio_diff
0,twitter_cluster10,0.01,0.499899,0.499899,0.000000e+00
1,twitter_cluster10,0.10,0.499899,0.499899,0.000000e+00
2,twitter_cluster11,0.01,0.428254,0.401297,2.695643e-02
3,twitter_cluster13,0.01,0.627929,0.627929,-1.575028e-08
4,twitter_cluster13,0.10,0.627895,0.627895,-6.057801e-09


In [8]:
small_cache_df = comparison_df[comparison_df['wsr'] == 0.1]
marginal_hits_better = len(small_cache_df[small_cache_df['miss_ratio_diff'] <= 0])
lama_better = len(small_cache_df[small_cache_df['miss_ratio_diff'] > 0])
print(f"Marginal Hits better: {marginal_hits_better}, Lama better: {lama_better}")
small_cache_df.sort_values(by='miss_ratio_diff', ascending=False)

Marginal Hits better: 22, Lama better: 10


,trace_name,wsr,marginal_hits_tuned_miss_ratio,lama_miss_ratio,miss_ratio_diff
58,twitter_cluster8,0.1,0.392689,0.362573,3.011540e-02
26,twitter_cluster34,0.1,0.188062,0.170062,1.799985e-02
57,twitter_cluster7,0.1,0.151797,0.135188,1.660966e-02
7,twitter_cluster18,0.1,0.051105,0.044505,6.600308e-03
21,twitter_cluster3,0.1,0.024139,0.021535,2.604257e-03
46,twitter_cluster49,0.1,0.095541,0.094115,1.426196e-03
18,twitter_cluster24,0.1,0.026156,0.025273,8.829226e-04
49,twitter_cluster51,0.1,0.008351,0.007835,5.165397e-04
19,twitter_cluster26,0.1,0.078177,0.078144,3.292578e-05
39,twitter_cluster41,0.1,0.088242,0.088239,2.654662e-06


In [7]:
# Analysis of the comparison results differentiated by WSR
print("=== LAMA vs Marginal-Hits-Tuned Comparison Analysis ===")
print(f"Total comparisons: {len(comparison_df)}")
print()

# Overall statistics
print("=== OVERALL STATISTICS ===")
print("Miss Ratio Difference Statistics (marginal-hits-tuned - lama):")
print(f"Mean: {comparison_df['miss_ratio_diff'].mean():.6f}")
print(f"Median: {comparison_df['miss_ratio_diff'].median():.6f}")
print(f"Std: {comparison_df['miss_ratio_diff'].std():.6f}")
print(f"Min: {comparison_df['miss_ratio_diff'].min():.6f}")
print(f"Max: {comparison_df['miss_ratio_diff'].max():.6f}")
print()

# Overall performance comparison
marginal_hits_better = (comparison_df['miss_ratio_diff'] < 0).sum()
lama_better = (comparison_df['miss_ratio_diff'] > 0).sum()
ties = (comparison_df['miss_ratio_diff'] == 0).sum()

print("Overall Performance Comparison:")
print(f"Marginal-Hits-Tuned better (lower miss ratio): {marginal_hits_better} times ({marginal_hits_better/len(comparison_df)*100:.1f}%)")
print(f"LAMA better (lower miss ratio): {lama_better} times ({lama_better/len(comparison_df)*100:.1f}%)")
print(f"Ties: {ties} times ({ties/len(comparison_df)*100:.1f}%)")
print()

# Analysis by WSR
print("=" * 60)
for wsr in sorted(comparison_df['wsr'].unique()):
    wsr_data = comparison_df[comparison_df['wsr'] == wsr]
    print(f"=== ANALYSIS FOR WSR = {wsr} ===")
    print(f"Number of comparisons: {len(wsr_data)}")
    print()
    
    print("Miss Ratio Difference Statistics (marginal-hits-tuned - lama):")
    print(f"Mean: {wsr_data['miss_ratio_diff'].mean():.6f}")
    print(f"Median: {wsr_data['miss_ratio_diff'].median():.6f}")
    print(f"Std: {wsr_data['miss_ratio_diff'].std():.6f}")
    print(f"Min: {wsr_data['miss_ratio_diff'].min():.6f}")
    print(f"Max: {wsr_data['miss_ratio_diff'].max():.6f}")
    print()
    
    # Performance comparison for this WSR
    wsr_marginal_better = (wsr_data['miss_ratio_diff'] < 0).sum()
    wsr_lama_better = (wsr_data['miss_ratio_diff'] > 0).sum()
    wsr_ties = (wsr_data['miss_ratio_diff'] == 0).sum()
    
    print(f"Performance Comparison for WSR = {wsr}:")
    print(f"Marginal-Hits-Tuned better: {wsr_marginal_better} times ({wsr_marginal_better/len(wsr_data)*100:.1f}%)")
    print(f"LAMA better: {wsr_lama_better} times ({wsr_lama_better/len(wsr_data)*100:.1f}%)")
    print(f"Ties: {wsr_ties} times ({wsr_ties/len(wsr_data)*100:.1f}%)")
    print()
    
    # Results by trace for this WSR
    print(f"Results by trace for WSR = {wsr}:")
    for trace in sorted(wsr_data['trace_name'].unique()):
        trace_data = wsr_data[wsr_data['trace_name'] == trace]
        trace_marginal_better = (trace_data['miss_ratio_diff'] < 0).sum()
        trace_lama_better = (trace_data['miss_ratio_diff'] > 0).sum()
        avg_diff = trace_data['miss_ratio_diff'].mean()
        print(f"  {trace}: Marginal-Hits-Tuned better {trace_marginal_better}/{len(trace_data)}, "
              f"LAMA better {trace_lama_better}/{len(trace_data)}, "
              f"Avg diff: {avg_diff:.6f}")
    print()
    print("=" * 60)

=== LAMA vs Marginal-Hits-Tuned Comparison Analysis ===
Total comparisons: 59

=== OVERALL STATISTICS ===
Miss Ratio Difference Statistics (marginal-hits-tuned - lama):
Mean: 0.004907
Median: 0.000000
Std: 0.014152
Min: -0.012836
Max: 0.061715

Overall Performance Comparison:
Marginal-Hits-Tuned better (lower miss ratio): 22 times (37.3%)
LAMA better (lower miss ratio): 25 times (42.4%)
Ties: 12 times (20.3%)

=== ANALYSIS FOR WSR = 0.004 ===
Number of comparisons: 1

Miss Ratio Difference Statistics (marginal-hits-tuned - lama):
Mean: 0.061715
Median: 0.061715
Std: nan
Min: 0.061715
Max: 0.061715

Performance Comparison for WSR = 0.004:
Marginal-Hits-Tuned better: 0 times (0.0%)
LAMA better: 1 times (100.0%)
Ties: 0 times (0.0%)

Results by trace for WSR = 0.004:
  twitter_cluster53: Marginal-Hits-Tuned better 0/1, LAMA better 1/1, Avg diff: 0.061715

=== ANALYSIS FOR WSR = 0.01 ===
Number of comparisons: 22

Miss Ratio Difference Statistics (marginal-hits-tuned - lama):
Mean: 0.01007